# 03 — Supervised Modeling: Predicting Clinical Endpoints

*How efficient are models based on microarray data combined with features of age and gender at predicting clinical endpoints?*

> **Reconstruction note.** The original supervised-modeling notebook for this MSc project was
> not recoverable — it resided only in a local Jupyter working directory that was not captured
> in backup. This notebook **reconstructs that stage to reproduce the methodology and results
> documented in the project manuscript**, not to introduce new analysis. It implements the three
> classifiers, the train/test protocol, the hyperparameter tuning and the class-imbalance handling
> described in the report. The pipeline was validated on a 120-row sample; **run it on the full
> cleaned dataset to regenerate the reported figures** (e.g. Random Forest High-Risk F1 ≈ 0.94;
> XGBoost strongest overall, ROC-AUC ≈ 0.72 for INSS Stage). Reported metrics are cited from the
> manuscript and may differ slightly under different random seeds.

**Input:** `merged_clinical_gene_expression_data_cleaned.csv` — samples × [clinical columns + ~38,945 log2 gene-expression features], already imputed and merged in notebook 01.

**Features used:** age, gender, and the gene-expression matrix (matching the manuscript's stated design; `MYCN.status` and `clinico.genetic.subgroup` are present in the file but deliberately excluded so performance reflects microarray + age + gender only).

**Endpoints (targets):** INSS Stage (multiclass), High Risk, Progression of Disease, Death from Disease (binary).

**Output:** per-endpoint accuracy / recall / F1 and ROC-AUC; a model-comparison table; ROC curves.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, classification_report)
from xgboost import XGBClassifier

# ----------------------------- configuration -----------------------------
DATA_PATH    = "../data/merged_clinical_gene_expression_data_cleaned.csv"  # or .csv.gz
TEST_SIZE    = 0.30          # 70/30 train-test split (per manuscript)
RANDOM_STATE = 42

# Tuning grids (kept to what the manuscript specifies).
RF_GRID  = {"n_estimators": [100, 200, 300], "max_depth": [None, 10, 20]}
SVM_GRID = {"C": [0.1, 1, 10], "gamma": ["scale", "auto"]}
XGB_DIST = {"n_estimators": [100, 200], "max_depth": [3, 5, 7], "learning_rate": [0.05, 0.1, 0.3]}

# Optional speed switch. The microarray matrix is ~39k features, so the SVM GridSearch is the
# slow step (minutes on the full data). Leave FEATURE_FILTER = None to stay faithful to the
# report (all cleaned genes). Set an integer K to keep only the K highest-variance genes for a
# faster run — clearly a deviation from the original, useful only for quick iteration.
FEATURE_FILTER = None


## 1. Load the cleaned data and identify column roles

The file is read straight from notebook 01's output. Pandas is told the metadata columns are
strings so the messy mixed-type encodings (see next step) load predictably.

In [ ]:
META = ["ID", "Gender", "Age", "clinico.genetic.subgroup", "MYCN.status",
        "INSS.Stage", "HighRisk", "Progression", "DeathFromDisease"]

df = pd.read_csv(DATA_PATH, low_memory=False)
gene_cols = [c for c in df.columns if c not in META]
print(f"rows: {df.shape[0]:,}  |  total columns: {df.shape[1]:,}  |  gene features: {len(gene_cols):,}")


## 2. Clean and encode

Two real data-quality issues in this file have to be fixed before modeling:

* **Inconsistent target encodings.** `INSS.Stage` stores the same class as both an integer and a
  string (e.g. `1` and `'1'`), which would otherwise fragment into separate classes. Casting to a
  stripped string merges them.
* **Categorical clinical fields.** `Gender` (M/F) and `HighRisk` (LR/HR) are mapped to 0/1
  (0 = Male, 1 = Female; 0 = low risk, 1 = high risk), matching the manuscript.

Gene values already sit on a log2 scale (~0.6–19.7), so no further log transform is applied; the
features are standardised later for the SVM only.

In [ ]:
# normalise the inconsistently typed multiclass target
df["INSS.Stage"] = df["INSS.Stage"].astype(str).str.strip()

# encode the categorical clinical features
gender = df["Gender"].map({"M": 0, "F": 1}).rename("Gender").astype("float32")

# feature matrix: age + gender + gene expression
genes = df[gene_cols].astype("float32")
if FEATURE_FILTER:
    top = genes.var().sort_values(ascending=False).head(int(FEATURE_FILTER)).index
    genes = genes[top]
    print(f"FEATURE_FILTER active: kept top {len(top):,} highest-variance genes")
X = pd.concat([df[["Age"]].astype("float32"), gender, genes], axis=1)

# targets
inss_le = LabelEncoder()
y = {
    "INSS Stage":         inss_le.fit_transform(df["INSS.Stage"]),
    "High Risk":          df["HighRisk"].map({"LR": 0, "HR": 1}).astype(int).values,
    "Progression":        df["Progression"].astype(int).values,
    "Death from Disease": df["DeathFromDisease"].astype(int).values,
}
MULTICLASS = {"INSS Stage": True, "High Risk": False, "Progression": False, "Death from Disease": False}

print("Feature matrix:", X.shape)
for k, v in y.items():
    vc = pd.Series(v).value_counts().sort_index().to_dict()
    print(f"  {k:<20s} classes/counts -> {vc}")


## 3. Helper functions

These keep the run robust on imbalanced clinical data — extreme class imbalance (the rare INSS
stages 2a/2b/3) is exactly what the manuscript flags as the limiting factor.

* `safe_split` — stratifies the 70/30 split where possible, falling back to a plain split when a
  class is too small to stratify.
* `cv_for` — sizes the cross-validation folds to the smallest class; if any class has a single
  member, tuning is skipped for that endpoint and default parameters are used (rather than crashing).
* `multiclass_auc` — one-vs-rest ROC-AUC averaged over the classes that are resolvable in the test set.
* `evaluate` — fits a model (tuned if possible), returns the standard metric set.

In [ ]:
def safe_split(X, yv, test=TEST_SIZE, seed=RANDOM_STATE):
    try:
        return train_test_split(X, yv, test_size=test, random_state=seed, stratify=yv)
    except ValueError:
        return train_test_split(X, yv, test_size=test, random_state=seed)

def cv_for(y_train):
    counts = pd.Series(y_train).value_counts().values
    m = counts.min()
    if m < 2:
        return None
    return StratifiedKFold(n_splits=int(min(5, m)), shuffle=True, random_state=RANDOM_STATE)

def multiclass_auc(y_true, proba, classes):
    aucs = []
    for i, c in enumerate(classes):
        yt = (y_true == c).astype(int)
        if 0 < yt.sum() < len(yt):
            aucs.append(roc_auc_score(yt, proba[:, i]))
    return float(np.mean(aucs)) if aucs else np.nan

def metrics(y_true, y_pred, proba, multiclass, classes=None):
    if multiclass:
        auc = multiclass_auc(y_true, proba, classes) if proba is not None else np.nan
        return dict(accuracy=accuracy_score(y_true, y_pred),
                    recall=recall_score(y_true, y_pred, average="weighted"),
                    f1=f1_score(y_true, y_pred, average="weighted"),
                    roc_auc=auc)
    auc = roc_auc_score(y_true, proba[:, 1]) if proba is not None else np.nan
    return dict(accuracy=accuracy_score(y_true, y_pred),
                recall=recall_score(y_true, y_pred),
                f1=f1_score(y_true, y_pred),
                roc_auc=auc)


## 4. Per-endpoint modeling: Random Forest, SVM, XGBoost

Each of the three classifiers is trained on every endpoint. Random Forest and XGBoost use the raw
features; the SVM uses standardised features. Class imbalance is handled with
`class_weight="balanced"` (RF, SVM) and `scale_pos_weight` (XGBoost, binary endpoints). Tuning
follows the manuscript: GridSearch for RF (`n_estimators`, `max_depth`) and SVM (`C`, `gamma`),
and a randomized search for XGBoost.

> **Compute note.** With ~39k features, the SVM GridSearch is the expensive step — expect several
> minutes per endpoint on the full data. Random Forest and XGBoost are quicker. Set `FEATURE_FILTER`
> above for a faster (non-faithful) pass while iterating.

In [ ]:
def run_all(name, yv, multiclass):
    Xtr, Xte, ytr, yte = safe_split(X, yv)
    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)
    cv = cv_for(ytr)
    classes = np.unique(ytr)
    scoring = "f1_weighted" if multiclass else "f1"
    rows = {}

    # --- Random Forest (raw features) ---
    rf = RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
    if cv is not None:
        rf = GridSearchCV(rf, RF_GRID, cv=cv, scoring=scoring, n_jobs=-1).fit(Xtr, ytr).best_estimator_
    else:
        rf = rf.fit(Xtr, ytr)
    rows["Random Forest"] = metrics(yte, rf.predict(Xte), rf.predict_proba(Xte), multiclass, classes)

    # --- SVM (standardised features) ---
    svm = SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)
    if cv is not None:
        svm = GridSearchCV(svm, SVM_GRID, cv=cv, scoring=scoring, n_jobs=-1).fit(Xtr_s, ytr).best_estimator_
    else:
        svm = svm.fit(Xtr_s, ytr)
    rows["SVM"] = metrics(yte, svm.predict(Xte_s), svm.predict_proba(Xte_s), multiclass, classes)

    # --- XGBoost (raw features) ---
    if multiclass:
        xgb = XGBClassifier(eval_metric="mlogloss", random_state=RANDOM_STATE, n_jobs=-1)
    else:
        pos = max((ytr == 1).sum(), 1)
        xgb = XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1,
                            scale_pos_weight=(ytr == 0).sum() / pos)
    if cv is not None:
        xgb = RandomizedSearchCV(xgb, XGB_DIST, n_iter=8, cv=cv, scoring=scoring,
                                 random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr).best_estimator_
    else:
        xgb = xgb.fit(Xtr, ytr)
    rows["XGBoost"] = metrics(yte, xgb.predict(Xte), xgb.predict_proba(Xte), multiclass, classes)

    return rows

records = []
for name, yv in y.items():
    print(f"Fitting endpoint: {name} ...")
    for model, m in run_all(name, yv, MULTICLASS[name]).items():
        records.append({"Endpoint": name, "Model": model, **{k: round(v, 3) for k, v in m.items()}})

results = pd.DataFrame(records)
results


## 5. Multi-output Random Forest (simultaneous prediction)

The manuscript also frames Random Forest as a single multi-output classifier predicting all four
endpoints at once. This mirrors that, using a stratified split on the High-Risk label.

In [ ]:
Y = np.column_stack([y["INSS Stage"], y["High Risk"], y["Progression"], y["Death from Disease"]])
Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=TEST_SIZE, random_state=RANDOM_STATE,
                                      stratify=y["High Risk"])
mo_rf = MultiOutputClassifier(
    RandomForestClassifier(n_estimators=300, class_weight="balanced",
                           random_state=RANDOM_STATE, n_jobs=-1)).fit(Xtr, Ytr)
P = mo_rf.predict(Xte)
mo = pd.DataFrame({
    "Endpoint": ["INSS Stage", "High Risk", "Progression", "Death from Disease"],
    "accuracy": [round(accuracy_score(Yte[:, i], P[:, i]), 3) for i in range(4)],
    "f1_weighted": [round(f1_score(Yte[:, i], P[:, i], average="weighted"), 3) for i in range(4)],
})
mo


## 6. Model comparison and ROC curves

Pivoted comparison across models, followed by ROC curves for the binary endpoints. Compare these
against the manuscript's reported figures — Random Forest strongest on High Risk, XGBoost the most
reliable overall, and all models limited on the rare INSS stages by class imbalance.

In [ ]:
pivot = results.pivot_table(index="Endpoint", columns="Model", values="f1").round(3)
print("F1 by endpoint and model:")
display(pivot)
print("\nROC-AUC by endpoint and model:")
display(results.pivot_table(index="Endpoint", columns="Model", values="roc_auc").round(3))


In [ ]:
# ROC curves for the binary endpoints, best model per endpoint (Random Forest shown)
binary = [k for k in y if not MULTICLASS[k]]
fig, axes = plt.subplots(1, len(binary), figsize=(4.2 * len(binary), 4))
for ax, name in zip(np.atleast_1d(axes), binary):
    Xtr, Xte, ytr, yte = safe_split(X, y[name])
    rf = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
    proba = rf.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(yte, proba)
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc_score(yte, proba):.2f}")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_title(name); ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()


## 7. Notes on interpretation

These results are reproductions of the analysis described in the project manuscript and should be
read as such — observations supported by the modeling, not new claims. Consistent with the report:

* **High Risk** is the most predictable endpoint; Random Forest and XGBoost both perform strongly.
* **XGBoost** is the most reliable model overall, though its advantage over the others is modest.
* The **rare INSS stages (2a, 2b, 3)** are limited by severe class imbalance across every model —
  the report identifies advanced resampling or additional clinical variables as the route to improve them.
* Age was used directly; the non-linear age grouping explored in the clustering notebook was, per
  the manuscript, not fully carried into the supervised stage.

Refer to the manuscript in `report/` for the full discussion and the originally reported metrics.